In [0]:
use catalog identifier(:catalog);

In [0]:
CREATE OR REPLACE TABLE gold.dim_date AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d AS calendar_date,
  date_format(d, 'EEEE') AS day_name,
  EXTRACT(WEEK FROM d) AS iso_week,
  dayofweek(d) IN (1, 7) AS is_weekend
FROM (
  SELECT explode(
    sequence(to_date('2026-04-01'), to_date('2026-04-30'), INTERVAL 1 DAY)
  ) AS d
)
ORDER BY calendar_date;

create or replace table gold.dim_stations as
select * from silver.stations;

CREATE OR REPLACE TABLE gold.dim_charger AS
SELECT
  c.* EXCEPT (last_changed_snapshot_date),
  s.city AS station_city
FROM silver.chargers c
LEFT JOIN silver.stations s
  ON c.station_id = s.station_id;

CREATE OR REPLACE TABLE gold.dim_member AS
SELECT
  CAST(ROW_NUMBER() OVER (ORDER BY m.member_id) AS BIGINT) AS member_sk,
  m.*
FROM silver.members m;

In [0]:
CREATE OR REPLACE TABLE gold.fact_sessions AS
WITH base AS (
  SELECT
    s.session_id,
    d.date_key,
    s.charger_id,
    c.station_id,
    s.member_id,
    s.status,
    s.time_band,
    s.energy_kwh,
    s.duration_minutes,
    s.idle_minutes,
    s.amount_paid,
    c.connector_type,
    m.member_sk,
    m.plan
  FROM silver.sessions s
  JOIN gold.dim_date d
    ON s.session_date = d.calendar_date
  LEFT JOIN silver.chargers c
    ON s.charger_id = c.charger_id
  LEFT JOIN gold.dim_member m
    ON s.member_id = m.member_id
   AND s.session_date >= m.effective_from
   AND (m.effective_to IS NULL OR s.session_date < m.effective_to)
),
faults AS (
  SELECT session_id, COUNT(*) AS fault_count
  FROM silver.session_faults
  GROUP BY session_id
),
priced AS (
  SELECT
    b.*,
    t.rate_per_kwh,
    t.free_idle_minutes,
    t.idle_fee_per_min,
    COALESCE(f.fault_count, 0) AS fault_count
  FROM base b
  LEFT JOIN silver.tariffs t
    ON b.connector_type = t.connector_type
   AND b.time_band = t.time_band
  LEFT JOIN faults f
    ON b.session_id = f.session_id
)
SELECT
  session_id,
  date_key,
  charger_id,
  station_id,
  member_id,
  status,
  time_band,
  energy_kwh,
  duration_minutes,
  idle_minutes,
  amount_paid,
  member_sk,

  CASE WHEN member_id IS NULL THEN 'GUEST' ELSE plan END AS plan_at_session,
  rate_per_kwh,
  CASE WHEN status = 'COMPLETED'
       THEN ROUND(energy_kwh * rate_per_kwh, 2)
       ELSE 0.00 END AS energy_charge,
  CASE WHEN status = 'COMPLETED'
       THEN ROUND(
              ROUND(energy_kwh * rate_per_kwh, 2) *
              CASE plan_at_session
                WHEN 'BASIC' THEN 0.00
                WHEN 'PRO'   THEN 0.10
                WHEN 'FLEET' THEN 0.15
                ELSE 0.00            
              END, 2)
       ELSE 0.00 END AS plan_discount,
  CASE WHEN status = 'COMPLETED'
       THEN ROUND(
              GREATEST(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min,
              2)
       ELSE 0.00 END AS idle_fee,
  CASE WHEN status = 'COMPLETED'
       THEN ROUND(energy_charge - plan_discount + idle_fee, 2)
       ELSE 0.00 END AS billed_amount,
  CASE WHEN status <> 'COMPLETED'
       THEN NULL
       WHEN ABS(COALESCE(amount_paid, 0) - billed_amount) <= 0.50
       THEN 'OK'
       ELSE 'MISMATCH' END AS billing_status,
  fault_count
FROM priced

In [0]:
CREATE OR REPLACE TABLE gold.agg_daily_station AS
WITH bounds AS (
  SELECT
    MIN(session_date) AS first_date,
    MAX(session_date) AS last_date
  FROM silver.sessions
),
dates AS (
  SELECT explode(sequence(first_date, last_date, INTERVAL 1 DAY)) AS session_date
  FROM bounds
),
stations AS (
  SELECT
    st.station_id,
    st.city,
    COUNT(c.charger_id) AS charger_count
  FROM gold.dim_stations st
  LEFT JOIN gold.dim_charger c
    ON st.station_id = c.station_id
  GROUP BY st.station_id, st.city
),
agg AS (
  SELECT
    d.calendar_date AS session_date,
    f.station_id,
    SUM(CASE WHEN f.status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_sessions,
    SUM(CASE WHEN f.status <> 'COMPLETED' THEN 1 ELSE 0 END) AS failed_sessions,
    SUM(CASE WHEN f.status = 'COMPLETED' THEN f.energy_kwh ELSE 0 END) AS energy_kwh,
    SUM(CASE WHEN f.status = 'COMPLETED' THEN f.billed_amount ELSE 0 END) AS billed_amount,
    SUM(CASE WHEN f.status = 'COMPLETED' THEN COALESCE(f.amount_paid, 0) ELSE 0 END) AS amount_paid,
    SUM(CASE WHEN f.billing_status = 'MISMATCH' THEN 1 ELSE 0 END) AS mismatch_sessions,
    SUM(COALESCE(f.fault_count, 0)) AS fault_count,
    SUM(CASE WHEN f.status = 'COMPLETED' THEN f.duration_minutes ELSE 0 END) AS completed_duration_minutes
  FROM gold.fact_sessions f
  JOIN gold.dim_date d
    ON f.date_key = d.date_key
  GROUP BY d.calendar_date, f.station_id
)
SELECT
  dt.session_date,
  st.station_id,
  st.city,
  CAST(COALESCE(a.completed_sessions, 0) AS BIGINT) AS completed_sessions,
  CAST(COALESCE(a.failed_sessions, 0) AS BIGINT) AS failed_sessions,
  CAST(COALESCE(a.energy_kwh, 0) AS DECIMAL(10,3)) AS energy_kwh,
  CAST(COALESCE(a.billed_amount, 0) AS DECIMAL(10,2)) AS billed_amount,
  CAST(COALESCE(a.amount_paid, 0) AS DECIMAL(10,2)) AS amount_paid,
  CAST(COALESCE(a.mismatch_sessions, 0) AS BIGINT) AS mismatch_sessions,
  CAST(COALESCE(a.fault_count, 0) AS BIGINT) AS fault_count,
  CAST(
    100 * COALESCE(a.completed_duration_minutes, 0)
      / NULLIF(st.charger_count * 1440, 0)
    AS DECIMAL(5,2)
  ) AS utilisation_pct
FROM dates dt
CROSS JOIN stations st
LEFT JOIN agg a
  ON dt.session_date = a.session_date
 AND st.station_id = a.station_id
ORDER BY st.station_id, dt.session_date;

In [0]:
CREATE OR REPLACE VIEW gold.v_kpi_summary AS
SELECT
  CAST(SUM(CASE WHEN status = 'COMPLETED' THEN 1 ELSE 0 END) AS BIGINT) AS completed_sessions,
  CAST(SUM(CASE WHEN status <> 'COMPLETED' THEN 1 ELSE 0 END) AS BIGINT) AS failed_sessions,
  CAST(
    100 * SUM(CASE WHEN status = 'COMPLETED' THEN 1 ELSE 0 END)
      / NULLIF(
          SUM(CASE WHEN status IN ('COMPLETED') OR status <> 'COMPLETED' THEN 1 ELSE 0 END),
          0)
    AS DECIMAL(5,2)
  ) AS success_rate_pct,
  CAST(SUM(CASE WHEN status = 'COMPLETED' THEN energy_kwh ELSE 0 END) AS DECIMAL(10,2)) AS total_energy_kwh,
  CAST(SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END) AS DECIMAL(10,2)) AS total_billed,
  CAST(SUM(CASE WHEN status = 'COMPLETED' THEN COALESCE(amount_paid, 0) ELSE 0 END) AS DECIMAL(10,2)) AS total_paid,
  CAST(
    SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END)
      - SUM(CASE WHEN status = 'COMPLETED' THEN COALESCE(amount_paid, 0) ELSE 0 END)
    AS DECIMAL(10,2)
  ) AS revenue_leakage,
  CAST(SUM(CASE WHEN billing_status = 'MISMATCH' THEN 1 ELSE 0 END) AS BIGINT) AS mismatch_sessions,
  CAST(COUNT(DISTINCT CASE WHEN status = 'COMPLETED' THEN member_id END) AS BIGINT) AS active_members
FROM gold.fact_sessions;


In [0]:
CREATE or replace table ops.reconciliation_results (
  run_ts        TIMESTAMP,
  check_name    STRING,
  source_value  DECIMAL(18,3),
  target_value  DECIMAL(18,3),
  status        STRING
);

INSERT INTO ops.reconciliation_results
WITH checks AS (
  SELECT
    'SESSION_COUNT' AS check_name,
    CAST((SELECT COUNT(DISTINCT session_id) FROM bronze.sessions) AS DECIMAL(18,3)) AS source_value,
    CAST(
      (SELECT COUNT(*) FROM silver.sessions)
      + (SELECT COUNT(*) FROM silver.quarantine_sessions)
      AS DECIMAL(18,3)
    ) AS target_value

  UNION ALL

  SELECT
    'ENERGY_KWH',
    CAST((SELECT COALESCE(SUM(energy_kwh), 0) FROM silver.sessions WHERE status = 'COMPLETED') AS DECIMAL(18,3)),
    CAST((SELECT COALESCE(SUM(energy_kwh), 0) FROM gold.agg_daily_station) AS DECIMAL(18,3))

  UNION ALL

  SELECT
    'CURRENT_MEMBERS',
    CAST((SELECT COUNT(DISTINCT member_id) FROM silver.members) AS DECIMAL(18,3)),
    CAST((SELECT COUNT(*) FROM gold.dim_member WHERE effective_to IS NULL) AS DECIMAL(18,3))
)
SELECT
  current_timestamp() AS run_ts,
  check_name,
  source_value,
  target_value,
  CASE WHEN source_value = target_value THEN 'PASS' ELSE 'FAIL' END AS status
FROM checks;




In [0]:
CREATE table if not exists ops.checkpoint_counts (
  checkpoint_name STRING,
  table_name      STRING,
  row_count       BIGINT,
  captured_at     TIMESTAMP
);

INSERT INTO ops.checkpoint_counts
SELECT
  :checkpoint_name AS checkpoint_name,
  table_name,
  row_count,
  current_timestamp() AS captured_at
FROM (
  SELECT 'bronze.sessions' AS table_name, COUNT(*) AS row_count FROM bronze.sessions
  UNION ALL
  SELECT 'bronze.members', COUNT(*) FROM bronze.members
  UNION ALL
  SELECT 'bronze.chargers', COUNT(*) FROM bronze.chargers
  UNION ALL
  SELECT 'silver.sessions', COUNT(*) FROM silver.sessions
  UNION ALL
  SELECT 'silver.quarantine_sessions', COUNT(*) FROM silver.quarantine_sessions
  UNION ALL
  SELECT 'silver.members', COUNT(*) FROM silver.members
  UNION ALL
  SELECT 'silver.chargers', COUNT(*) FROM silver.chargers
  UNION ALL
  SELECT 'gold.fact_sessions', COUNT(*) FROM gold.fact_sessions
  UNION ALL
  SELECT 'gold.agg_daily_station', COUNT(*) FROM gold.agg_daily_station
)
ORDER BY table_name;

SELECT * FROM ops.checkpoint_counts
ORDER BY captured_at DESC, table_name;
